Fluxo principal de Demonstração RAG

*Baixar dependencias*

In [ ]:
%pip install --quiet --upgrade "langchain[aws]"
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph
%pip install --quiet --upgrade python-dotenv
%pip install -qU langchain-core


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

LANGSMITH_TRACING=True
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT=os.getenv("LANGSMITH_PROJECT")

In [9]:
# Ensure your AWS credentials are configured

from langchain.chat_models import init_chat_model
from langchain_aws import BedrockEmbeddings

converse_model = init_chat_model("amazon.nova-micro-v1:0", model_provider="bedrock_converse")
embeddings_model = BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0")

In [10]:
%pip install -qU langchain-core

Note: you may need to restart the kernel to use updated packages.


In [11]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings_model)

Agora começa o RAG

In [12]:
import bs4
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# Load and chunk contents of the blog
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()